# Learning Management System Analytics Pipeline

## Phase 1: Data Ingestion

### Objective

The objective of this notebook is to:

- Read raw CSV datasets from Databricks Volume
- Inspect the schema
- Perform initial data profiling
- Check data quality issues
- Prepare data for the Bronze Layer

---

**Author:** Ravi

**Organization:** Celebal Technologies Internship

**Architecture:** Medallion (Bronze → Silver → Gold)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
BASE_PATH = "/Volumes/workspace/default/lms_data"

LEARNERS_PATH = f"{BASE_PATH}/learners.csv"
COURSES_PATH = f"{BASE_PATH}/courses.csv"
ENROLMENT_PATH = f"{BASE_PATH}/enrolment_activity.csv"

In [0]:
learners_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(LEARNERS_PATH)
)

courses_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(COURSES_PATH)
)

enrolment_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ENROLMENT_PATH)
)

In [0]:
print("Learners :", learners_df.count())
print("Courses :", courses_df.count())
print("Enrolments :", enrolment_df.count())

Learners : 500
Courses : 60
Enrolments : 2000


In [0]:
learners_df.printSchema()

root
 |-- learner_id: string (nullable = true)
 |-- learner_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)



In [0]:
courses_df.printSchema()

root
 |-- course_id: string (nullable = true)
 |-- course_title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- instructor_id: string (nullable = true)
 |-- instructor_name: string (nullable = true)
 |-- duration_hours: integer (nullable = true)
 |-- difficulty_level: string (nullable = true)
 |-- price_inr: integer (nullable = true)



In [0]:
enrolment_df.printSchema()

root
 |-- enrolment_id: string (nullable = true)
 |-- learner_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- enrol_date: date (nullable = true)
 |-- expected_completion_date: date (nullable = true)
 |-- actual_completion_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- progress_pct: integer (nullable = true)
 |-- last_activity_date: date (nullable = true)
 |-- assessment_score: double (nullable = true)
 |-- attempts: integer (nullable = true)
 |-- feedback_rating: integer (nullable = true)
 |-- certificate_issued: string (nullable = true)



In [0]:
display(learners_df)

learner_id,learner_name,email,phone_number,city,registration_date,subscription_type
LRN0001,Ananya Sharma,ananya.sharma.1@gmail.com,9433218196,Mumbai,2022-04-06,Free
LRN0002,Sachin Pillai,sachin.pillai.2@gmail.com,9083863794,Indore,2022-06-13,Premium
LRN0003,Suresh Mishra,suresh.mishra.3@rediffmail.com,9235116155,Chennai,2022-02-14,Premium
LRN0004,Vijay Kumar,vijay.kumar.4@gmail.com,9618495931,Surat,2022-08-22,Premium
LRN0005,Priya Chatterjee,priya.chatterjee.5@yahoo.com,9316475255,Surat,2022-10-01,Premium
LRN0006,Karan Pillai,karan.pillai.6@outlook.com,9283276483,Bhopal,2022-02-27,Free
LRN0007,Aditya Bose,aditya.bose.7@hotmail.com,9564139537,Surat,2023-04-15,Free
LRN0008,Divya Yadav,divya.yadav.8@yahoo.com,9884969653,Jaipur,2023-05-21,Free
LRN0009,Rohan Chatterjee,rohan.chatterjee.9@hotmail.com,9122691669,Jaipur,2022-09-15,Enterprise
LRN0010,Aarav Desai,aarav.desai.10@gmail.com,9184514627,Chandigarh,2022-09-27,Enterprise


In [0]:
display(courses_df)

course_id,course_title,category,instructor_id,instructor_name,duration_hours,difficulty_level,price_inr
CRS001,Python for Data Science,Data Science,INS009,Suresh Gupta,15,Intermediate,2986
CRS002,Machine Learning Fundamentals,AI & ML,INS006,Priya Singh,10,Beginner,223
CRS003,Deep Learning with TensorFlow,AI & ML,INS004,Sunita Rao,10,Advanced,13612
CRS004,React.js Complete Guide,Web Development,INS014,Pooja Mishra,30,Advanced,6867
CRS005,Node.js Backend Development,Web Development,INS007,Vikas Mehta,20,Beginner,119
CRS006,HTML & CSS Mastery,Web Development,INS015,null,5,Intermediate,2276
CRS007,Business Analytics Essentials,Business,INS003,Arjun Patel,20,Advanced,10604
CRS008,Project Management Professional,Business,INS003,Arjun Patel,40,Intermediate,4049
CRS009,Digital Marketing Strategy,Business,INS004,Sunita Rao,10,Advanced,8712
CRS010,UI/UX Design Principles,Design,INS005,Deepak Joshi,10,Beginner,133


In [0]:
display(enrolment_df)

enrolment_id,learner_id,course_id,enrol_date,expected_completion_date,actual_completion_date,status,progress_pct,last_activity_date,assessment_score,attempts,feedback_rating,certificate_issued
ENR00460,LRN0376,CRS001,2024-01-13,2024-02-12,2024-02-21,Completed,100,2024-02-21,79.22,1,2,Yes
ENR00681,LRN0486,CRS055,2024-01-16,2024-01-26,2024-01-19,Completed,100,2024-01-19,56.62,1,3,No
ENR01477,LRN0352,CRS044,2024-03-16,2024-06-04,2024-03-31,Completed,100,2024-03-31,93.98,1,4,Yes
ENR00554,LRN0474,CRS035,2024-02-01,2024-04-21,null,Dropped,8,2024-02-07,null,1,null,No
ENR00640,LRN0240,CRS038,2024-02-11,2024-03-12,2024-03-01,Completed,100,2024-03-01,81.21,2,3,Yes
ENR00309,LRN0457,CRS012,2024-03-09,2024-04-18,null,Dropped,28,2024-03-10,null,1,null,No
ENR00406,LRN0316,CRS060,2024-02-22,2024-03-23,2024-03-27,Completed,100,2024-03-27,96.0,1,4,Yes
ENR00933,LRN0309,CRS042,2024-03-06,2024-03-16,null,Dropped,11,2024-03-09,null,1,null,No
ENR01153,LRN0078,CRS012,2024-02-06,2024-03-17,null,In Progress,15,2024-03-27,null,1,null,No
ENR00810,LRN0424,CRS039,2024-01-30,2024-04-19,null,Dropped,35,2024-02-10,null,2,null,No


# Data Profiling & Data Quality Assessment

## Objective

Before building the Medallion Architecture, it is important to understand the quality of the source datasets.

The following checks will be performed:

- Record Count
- Column Count
- Null Value Analysis
- Duplicate Detection
- Summary Statistics
- Distinct Value Analysis
- Data Quality Report

In [0]:
print("="*60)
print("DATASET SHAPE")
print("="*60)

datasets = {
    "Learners": learners_df,
    "Courses": courses_df,
    "Enrolments": enrolment_df
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(df.columns)}")

DATASET SHAPE

Learners
Rows    : 500
Columns : 7

Courses
Rows    : 60
Columns : 8

Enrolments
Rows    : 2000
Columns : 13


In [0]:
from pyspark.sql.functions import col, count, when

def null_count(df):

    return df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)

        for c in df.columns
    ])

print("Learners")
display(null_count(learners_df))

print("Courses")
display(null_count(courses_df))

print("Enrolments")
display(null_count(enrolment_df))

Learners


learner_id,learner_name,email,phone_number,city,registration_date,subscription_type
0,0,0,0,0,0,0


Courses


course_id,course_title,category,instructor_id,instructor_name,duration_hours,difficulty_level,price_inr
0,0,0,0,6,0,0,0


Enrolments


enrolment_id,learner_id,course_id,enrol_date,expected_completion_date,actual_completion_date,status,progress_pct,last_activity_date,assessment_score,attempts,feedback_rating,certificate_issued
0,0,0,0,0,1179,0,0,330,1179,0,1082,0


In [0]:
print("="*50)
print("Duplicate Check")
print("="*50)

print("Learners")
print(
    learners_df.count()
    -
    learners_df.dropDuplicates().count()
)

print("Courses")
print(
    courses_df.count()
    -
    courses_df.dropDuplicates().count()
)

print("Enrolments")
print(
    enrolment_df.count()
    -
    enrolment_df.dropDuplicates().count()
)

Duplicate Check
Learners
0
Courses
0
Enrolments
10


In [0]:
display(
    enrolment_df.describe()
)

summary,enrolment_id,learner_id,course_id,status,progress_pct,assessment_score,attempts,feedback_rating,certificate_issued
count,2000,2000,2000,2000,2000,821,2000,918,2000
mean,null,null,null,null,58.9435,71.2383556638247,1.265,3.0446623093681917,null
stddev,null,null,null,null,39.88542003183076,15.740521703945312,0.5512957132807622,1.4092577709215717,null
min,ENR00001,LRN0001,CRS001,Completed,0,45.2,1,1,No
max,ENR01990,LRN0500,CRS060,Not Started,100,99.78,3,5,Yes


In [0]:
display(

    enrolment_df.groupBy(
        "status"
    ).count()

)

status,count
Not Started,185
Dropped,415
In Progress,579
Completed,821


In [0]:
display(

learners_df.groupBy(
    "subscription_type"
).count()

)

subscription_type,count
Premium,138
Basic,161
Free,161
Enterprise,40


In [0]:
display(

courses_df.groupBy(
    "difficulty_level"
).count()

)

difficulty_level,count
Advanced,21
Intermediate,19
Beginner,20


In [0]:
display(

courses_df.groupBy(
    "category"
).count()

)

category,count
Web Development,8
Data Science,9
Cloud Computing,7
AI & ML,7
Design,8
Mobile Development,6
Cybersecurity,6
Business,9


# Data Quality Findings

## Learners Dataset

- No duplicate records
- Primary Key available
- Registration Date successfully parsed

---

## Courses Dataset

- Missing instructor names present
- No duplicate course records

---

## Enrolment Dataset

- Duplicate records exist
- Missing completion dates present
- Missing assessment scores
- Missing feedback ratings
- Missing last activity dates

---

## Conclusion

The datasets contain expected real-world data quality issues that will be resolved during the Silver Layer of the Medallion Architecture.